# Vicuña · Miembros nuevos para el ensamble, ronda 2 (T20)

La mezcla de 30 (envío 59028408, público 0.50332) mejoró al sumar modelos distintos. Tres más, todos sobre `lead`:

- `unet40`: la U-Net con 40 épocas.
- `unet64`: la U-Net con ancho 64 (25 épocas).
- `ancho96e40`: la red de ancho 96 con 40 épocas.

Cada una: 5 semillas en los cortes A y B (guarda las predicciones) y reentrenamiento con todos los datos (pesos y CSV).

**Regla de envío, fijada antes de correr** (se calcula fuera con las predicciones de t09e y t09h, pesos iguales por grupo de 5): frente a la mezcla de 30 (r1, r2, lead, unet, ancho96, ep40) se comparan +unet40, +unet64, +ancho96e40 y +las tres. Se elige la de mayor media en A y B y **se envía solo si supera a la de 30 en ≥ 0.001 en A y en B**. Entrada solo t-2, t-1, t.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

In [ ]:
# COPIA EXACTA de train/validacion.py: no editar aquí
"""
Validación temporal con la métrica del concurso (T02 · HU-09). Solo NumPy.

Corte temporal fijo: validación = últimas dos semanas de junio (desde 2024-06-16 00 UTC);
entrenamiento = todo lo anterior. Nunca se hace split aleatorio.

Hay dos scores y no son comparables entre sí:
  score_local    el del starter: cada RMSE se divide entre el RMSE de la climatología de train
                 medido en la propia validación (climatología = 0). Todas las horas.
  score_oficial  fórmula del leaderboard, con sus constantes fijas, sobre las horas que Kaggle
                 evalúa (00, 03, ..., 21 UTC). Es el que se usa para comparar experimentos.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_validacion.py comprueba que ambas sean iguales.
"""
import numpy as np

VARS = ("RH", "T", "U", "V")
INICIO_VAL = "2024061600"
ESCALAS_OFICIALES = np.array([25.360, 2.0652, 2.6776, 2.0876])
HORAS_EVALUADAS = tuple(range(0, 24, 3))


def horas(times):
    """Hora UTC (0-23) de cada marca 'YYYYMMDDHH'."""
    return np.array([int(str(t)[-2:]) for t in times])


def corte_temporal(times, inicio_val=INICIO_VAL):
    """Máscaras booleanas (train, val): val son las marcas desde `inicio_val` en adelante."""
    val = np.asarray(times).astype(str) >= inicio_val
    trn = ~val
    if not trn.any() or not val.any():
        raise ValueError(f"El corte {inicio_val} deja vacío el entrenamiento o la validación.")
    return trn, val


def climatologia(Y, times):
    """Media por hora UTC: (24, filas, columnas, variables). Exige datos en las 24 horas."""
    h = horas(times)
    faltan = sorted(set(range(24)) - set(h.tolist()))
    if faltan:
        raise ValueError(f"No hay datos para las horas UTC {faltan}.")
    return np.stack([Y[h == k].mean(0) for k in range(24)])


def rmse_por_variable(pred, real):
    """RMSE de cada variable sobre todas las horas y celdas. Entradas (N, filas, columnas, 4)."""
    if pred.shape != real.shape:
        raise ValueError(f"Formas distintas: pred {pred.shape} y real {real.shape}.")
    rmse = np.sqrt(((pred - real) ** 2).mean(axis=(0, 1, 2)))
    if not np.isfinite(rmse).all():
        raise ValueError("Hay NaN o infinitos en la predicción o en el valor real.")
    return rmse


def score(rmse, escalas):
    """1 - media de rmse/escala; las cuatro variables pesan igual."""
    return float(1 - (rmse / escalas).mean())


def escala_climatologia(Y, times, inicio_val=INICIO_VAL):
    """RMSE en validación de la climatología calculada solo con train (denominador de score_local)."""
    t = np.asarray(times).astype(str)
    trn, val = corte_temporal(t, inicio_val)
    clim = climatologia(Y[trn], t[trn])
    return rmse_por_variable(clim[horas(t[val])], Y[val])


def validar(pred_val, Y, times, inicio_val=INICIO_VAL):
    """
    Evalúa una predicción de la validación.

    pred_val : (N_val, 70, 35, 4) en unidades físicas, en el orden de las horas de validación.
    Y, times : target y marcas de TODO train.npz (de ahí salen el corte y la climatología).
    """
    t = np.asarray(times).astype(str)
    _, val = corte_temporal(t, inicio_val)
    real, t_val = Y[val], t[val]
    rmse = rmse_por_variable(pred_val, real)
    evaluadas = np.isin(horas(t_val), HORAS_EVALUADAS)
    rmse_3h = rmse_por_variable(pred_val[evaluadas], real[evaluadas])
    return {
        "score_local": score(rmse, escala_climatologia(Y, t, inicio_val)),
        "rmse": {v: float(r) for v, r in zip(VARS, rmse)},
        "score_oficial": score(rmse_3h, ESCALAS_OFICIALES),
        "rmse_horas_evaluadas": {v: float(r) for v, r in zip(VARS, rmse_3h)},
        "horas_val": int(val.sum()),
        "horas_evaluadas": int(evaluadas.sum()),
        "primera": str(min(t_val)),
        "ultima": str(max(t_val)),
    }

In [ ]:
def windows(X, t, n=3):
    """Ventana de n horas (t-n+1 .. t). Si falta una hora, se repite la hora t (como en el starter)."""
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), n) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        for j, lag in enumerate(range(n - 1, -1, -1)):
            idx = pos.get(k - pd.Timedelta(hours=lag), None)
            out[i, j] = X[idx] if idx is not None else X[i]
    return out   # (N, n, 17, 13, 4)

WTR = {n: windows(Xtr, ttr, n) for n in (3, 6)}
WTE = {n: windows(Xte, tte, n) for n in (3, 6)}
print({n: (WTR[n].shape, WTE[n].shape) for n in WTR})

In [ ]:
import json, platform
QUICK = False                      # True: prueba corta de que todo corre (en CPU)
assert QUICK or torch.cuda.is_available(), 'Este notebook necesita GPU'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
EQUIPO = torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU'
print(EQUIPO, '·', platform.python_version(), np.__version__, torch.__version__)
# Dos cortes temporales: A (el de T02) y B (1-15 jun, sin mirar nada posterior)
VENTANAS = {'A': dict(usar=np.ones(len(ttr), bool), ini='2024061600'),
            'B': dict(usar=ttr < '2024061600', ini='2024060100')}
for k, v in VENTANAS.items():
    v['trn'] = v['usar'] & (ttr < v['ini']); v['val'] = v['usar'] & (ttr >= v['ini'])
    print(f"ventana {k}: train {v['trn'].sum()} h · val {v['val'].sum()} h · {ttr[v['val']][0]} → {ttr[v['val']][-1]}")
def to_t(a): return torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)
ESC = to_t(ESCALAS_OFICIALES)

In [ ]:
# Grilla WRF dentro de la grilla GFS (como en el starter): GFS fila 0 = norte, WRF y DEM fila 0 = sur
glat, glon = st['gfs_lats'], st['gfs_lons']
wlat, wlon = st['wrf_lats'], st['wrf_lons']
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = to_t(np.stack([gx, gy], -1)[None])                      # (1,70,35,2)
DEM = to_t((dem - dem.mean()) / dem.std())[None, None]

def media_local(a, k):
    """Media en una ventana k x k con borde reflejado."""
    p = np.pad(a, k // 2, mode='reflect')
    return np.lib.stride_tricks.sliding_window_view(p, (k, k)).mean((-1, -2))

def relieve(dem):
    """Pendiente (norte, este, magnitud), orientación (sin, cos) y relieve relativo (~15 y ~45 km)."""
    z = dem.astype(np.float64)
    dn, de = np.gradient(z)                                    # fila crece hacia el norte, columna al este
    mag = np.hypot(dn, de)
    unit = np.where(mag > 0, mag, 1.0)
    capas = [dn, de, mag, de / unit, dn / unit, z - media_local(z, 5), z - media_local(z, 15)]
    return np.stack([(c - c.mean()) / c.std() for c in capas])

TERRENO = to_t(relieve(dem))[None]                             # (1,7,70,35)

def interpolar(a):
    """(B,C,17,13) GFS -> (B,C,70,35) WRF, bilineal."""
    return F.grid_sample(a, GRID.expand(len(a), -1, -1, -1), align_corners=True)

class Bloque(nn.Module):
    def __init__(self, w, d, p=0.0):
        super().__init__()
        self.p = p
        self.f = nn.Sequential(nn.Conv2d(w, w, 3, padding=d, dilation=d), nn.GELU(),
                               nn.Conv2d(w, w, 3, padding=d, dilation=d))
    def forward(self, x):
        return F.gelu(x + F.dropout2d(self.f(x), self.p, self.training))

class Net(nn.Module):
    """Red de la ronda 1 con ancho, bloques, codificador, dropout y número de horas configurables."""
    def __init__(self, skip=False, terreno=False, profundo=False, w=64,
                 bloques=(1, 2, 4, 1), enc_prof=False, dropout=0.0, lags=3, lead='ciclica'):
        super().__init__()
        self.skip, self.terreno, self.lead = skip, terreno, lead
        nh = {'ciclica': 2, 'lineal': 3, 'onehot': 24}[lead]
        self.enc = nn.Sequential(nn.Conv2d(4 * lags + nh, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.enc2 = nn.Sequential(Bloque(w, 1, dropout), Bloque(w, 1, dropout)) if enc_prof else nn.Identity()
        c = w + 9 + (4 * lags if skip else 0) + (7 if terreno else 0)
        if profundo:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     *[Bloque(w, d, dropout) for d in bloques], nn.Conv2d(w, 4, 1))
        else:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     nn.Conv2d(w, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, 4, 1))
    def forward(self, x, h):                     # x: (B,lags,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, -1, 17, 13)
        ang = 2 * np.pi * h / 24
        if self.lead == 'onehot':                # hora desde el inicio del WRF (00 UTC), un canal por hora
            hh = F.one_hot(h.long(), 24).float()
        else:
            hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)
            if self.lead == 'lineal':
                hh = torch.cat([hh, (h / 23 * 2 - 1)[:, None]], 1)
        hh = hh[..., None, None].expand(B, hh.shape[1], 17, 13)
        partes = [interpolar(self.enc2(self.enc(torch.cat([x, hh], 1)))), DEM.expand(B, -1, -1, -1),
                  self.static.expand(B, -1, -1, -1)]
        if self.skip:
            partes.append(interpolar(x))
        if self.terreno:
            partes.append(TERRENO.expand(B, -1, -1, -1))
        return self.dec(torch.cat(partes, 1)).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada

class UNet(nn.Module):
    """U-Net a 3 km: GFS interpolado (3 horas + hora) + DEM + mapa estático. Misma interfaz que Net."""
    def __init__(self, w=48, dropout=0.1, lags=3, **_):
        super().__init__()
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        def par(a, b):
            return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.GELU(), nn.Dropout2d(dropout),
                                 nn.Conv2d(b, b, 3, padding=1), nn.GELU())
        self.d1 = par(4 * lags + 3 + 9, w); self.d2 = par(w, 2 * w); self.d3 = par(2 * w, 4 * w)
        self.u2 = par(4 * w + 2 * w, 2 * w); self.u1 = par(2 * w + w, w)
        self.fin = nn.Conv2d(w, 4, 1)
    def forward(self, x, h):                     # x: (B,3,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        g = interpolar(x.permute(0, 1, 4, 2, 3).reshape(B, -1, 17, 13))
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang), h / 23 * 2 - 1], 1)[..., None, None].expand(B, 3, 70, 35)
        e1 = self.d1(torch.cat([g, hh, DEM.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1))
        e2 = self.d2(F.avg_pool2d(e1, 2, ceil_mode=True)); e3 = self.d3(F.avg_pool2d(e2, 2, ceil_mode=True))
        sube = lambda a, ref: F.interpolate(a, size=ref.shape[-2:], mode='bilinear', align_corners=False)
        d = self.u2(torch.cat([sube(e3, e2), e2], 1))
        d = self.u1(torch.cat([sube(d, e1), e1], 1))
        return self.fin(d).permute(0, 2, 3, 1)

In [ ]:
BASE = {'w': 64, 'bloques': [1, 2, 4, 1], 'enc_prof': False, 'dropout': 0.0, 'wd': 0.0001, 'lags': 3, 'skip': False, 'residuo': 'crudo', 'terreno': False, 'profundo': True, 'peso_mj': 1, 'epocas': 25, 'lead': 'ciclica'}      # ronda 1

def clim_ponderada(A, t, peso):
    """Media por hora UTC en la que las horas de mayo y junio pesan `peso` (1 = climatología de T02)."""
    if peso == 1:
        return climatologia(A, t)
    h = horas(t)
    w = np.where(np.array([s[4:6] in ('05', '06') for s in t]), float(peso), 1.0)
    return np.stack([np.average(A[h == k], axis=0, weights=w[h == k]) for k in range(24)]).astype(np.float32)

def datos_de(mascara, cfg):
    """Normalización y climatologías SOLO con las horas de `mascara`."""
    t = ttr[mascara]
    mu_x = Xtr[mascara].mean((0, 1, 2)); sd_x = Xtr[mascara].std((0, 1, 2))
    clim = clim_ponderada(Ytr[mascara], t, cfg['peso_mj'])
    clim_gfs = clim_ponderada(Xtr[mascara], t, cfg['peso_mj'])       # (24,17,13,4)
    anom_sd = (Ytr[mascara] - clim[horas(t)]).std((0, 1, 2))
    D = {'cfg': cfg, 'CLIM': to_t(clim), 'ASD': to_t(anom_sd), 'CLIM_GFS': to_t(clim_gfs),
         'stats': dict(mu_x=mu_x, sd_x=sd_x, clim=clim, anom_sd=anom_sd, clim_gfs=clim_gfs)}
    return D

def preparar_x(D, W, t):
    """Entrada normalizada, hora y término base del residuo (en anomalías normalizadas de WRF)."""
    s = D['stats']
    x = to_t((W - s['mu_x']) / s['sd_x'])
    h = to_t(horas(t)).long()
    modo = D['cfg']['residuo']
    if modo is None:
        return x, h, None
    partes = []
    for i in range(0, len(W), 256):
        gfs = to_t(W[i:i + 256, -1])                                   # hora t, unidades físicas
        hb = h[i:i + 256]
        if modo == 'anomalia':     # pred = clim_WRF + (GFS - clim_GFS) interpolado + red
            b = interpolar((gfs - D['CLIM_GFS'][hb]).permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        elif modo == 'crudo':      # pred = GFS interpolado + red
            b = interpolar(gfs.permute(0, 3, 1, 2)).permute(0, 2, 3, 1) - D['CLIM'][hb]
        else:
            raise ValueError(modo)
        partes.append(b / D['ASD'])
    return x, h, torch.cat(partes)

def salida(model, x, h, b):
    p = model(x, h.float())
    return p if b is None else p + b

def red_de(cfg):
    if cfg.get('arq') == 'unet':
        return UNet(w=cfg['w'], dropout=cfg['dropout'], lags=cfg['lags']).to(dev)
    return Net(**{k: cfg[k] for k in ('skip', 'terreno', 'profundo', 'w', 'bloques', 'enc_prof', 'dropout', 'lags', 'lead')}).to(dev)

def entrenar(D, tensores, epocas, semilla, lote=32):
    x, h, b, y = tensores
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    rng = np.random.default_rng(semilla)
    model = red_de(D['cfg'])
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=D['cfg']['wd'])
    n = len(x); pasos = epocas * int(np.ceil(n / lote))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=pasos)
    for ep in range(epocas):
        model.train()
        orden = torch.tensor(rng.permutation(n), device=dev)
        for s in range(0, n, lote):
            i = orden[s:s + lote]
            pred = salida(model, x[i], h[i], None if b is None else b[i])
            mse = ((pred - y[i]) ** 2).mean(dim=(0, 1, 2)) * (D['ASD'] / ESC) ** 2
            loss = torch.sqrt(mse + 1e-8).mean()       # pérdida 'rmse' de T08
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model

def tensores_de(D, mascara):
    x, h, b = preparar_x(D, WTR[D['cfg']['lags']][mascara], ttr[mascara])
    y = (to_t(Ytr[mascara]) - D['CLIM'][h]) / D['ASD']
    return x, h, b, y

def predecir_lote(model, D, W, t, bs=128):
    x, h, b = preparar_x(D, W, t); model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), bs):
            s = slice(i, i + bs)
            p = salida(model, x[s], h[s], None if b is None else b[s])
            out.append((D['CLIM'][h[s]] + p * D['ASD']).cpu().numpy())
    P = np.concatenate(out)
    P[..., 0] = np.clip(P[..., 0], 0, 100)
    return P

def medir(P, k):
    v = VENTANAS[k]
    r = validar(P, Ytr[v['usar']], ttr[v['usar']], inicio_val=v['ini'])
    return {'score_oficial': r['score_oficial'], 'score_local': r['score_local'], 'rmse_3h': r['rmse_horas_evaluadas']}

def linea(etiqueta, m):
    v = ' '.join(f'{k} {x:.3f}' for k, x in m['rmse_3h'].items())
    print(f"{etiqueta:34s} oficial {m['score_oficial']:.5f} · local {m['score_local']:.5f} · {v}", flush=True)

In [ ]:
LEAD = {**BASE, 'dropout': 0.1, 'lead': 'lineal'}
CFG = {'unet40': {**LEAD, 'arq': 'unet', 'w': 48, 'epocas': 40}, 'unet64': {**LEAD, 'arq': 'unet', 'w': 64},
       'ancho96e40': {**LEAD, 'w': 96, 'epocas': 40}}
N = 2 if QUICK else 5
if QUICK:
    _tensores = tensores_de
    def tensores_de(D, mascara):
        m = mascara.copy(); m[np.where(m)[0][64:]] = False
        return _tensores(D, m)
EP = lambda cfg: 1 if QUICK else cfg['epocas']
INICIO = time.time()
sample = pd.read_csv(find('sample_submission.csv'))
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti = parts[0].map(pos).values; r = parts[1].astype(int).values; c = parts[2].astype(int).values
os.makedirs('models', exist_ok=True)
res = {}
for nombre, cfg in CFG.items():
    for k in VENTANAS:
        v = VENTANAS[k]; D = datos_de(v['trn'], cfg); T = tensores_de(D, v['trn'])
        P = [predecir_lote(entrenar(D, T, EP(cfg), s), D, WTR[cfg['lags']][v['val']], ttr[v['val']]) for s in range(N)]
        E = np.mean(P, 0)
        res[f'{nombre}|{k}'] = dict(ens=medir(E, k), solas=[medir(p, k)['score_oficial'] for p in P])
        np.savez_compressed(f'val_{nombre}_{k}.npz', times=ttr[v['val']], ens=E.astype(np.float32))
        linea(f'{k} ensamble {nombre} ({time.time() - INICIO:.0f}s)', res[f'{nombre}|{k}']['ens'])
        del D, T, P; torch.cuda.empty_cache()
        json.dump(res, open('resultados_div2.json', 'w'), indent=1)
    todo = np.ones(len(ttr), bool); D = datos_de(todo, cfg); T = tensores_de(D, todo); suma = 0
    for s in range(N):
        mod = entrenar(D, T, EP(cfg), s)
        suma = suma + predecir_lote(mod, D, WTE[cfg['lags']], tte)
        torch.save(mod.state_dict(), f'models/{nombre}_s{s}.pt')
    np.savez(f'models/estadisticas_{nombre}.npz', **D['stats'])
    json.dump(cfg, open(f'models/config_{nombre}.json', 'w'))
    sub = pd.DataFrame({'id': sample['id']})
    for j, v_ in enumerate(VARS):
        sub[v_] = (suma / N)[ti, r, c, j].round(3)
    assert len(sub) == len(sample) and np.isfinite(sub[list(VARS)].values).all()
    sub.to_csv(f'submission_{nombre}.csv', index=False)
    print(f'submission_{nombre}.csv', sub.shape, f'· {time.time() - INICIO:.0f}s', flush=True)
    del D, T; torch.cuda.empty_cache()
print('terminado', f'{time.time() - INICIO:.0f}s')